# Exercise 10- Stacking

In this exercise you will implement an ensemble method by learning a stacked regressor.

In the event of a persistent problem, do not hesitate to contact the course instructor under

- maurice.wenig@uni-jena.de

### Submission
- Deadline of submission: 22.06.2026 23:59
- Submission on [moodle page](https://moodle.uni-jena.de/course/view.php?id=76636)


# The Dataset

We will use a real world dataset used for predicting the [quality of red wine](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009).
Altough the quality is a discrete value between 0 and 10, we interpret it as a regression task. 

### Task 1

Load the dataset stored in `dataset.csv` and split it into `x` and `y`.

In [36]:
# load data
import numpy as np
import polars as pl

df = pl.read_csv("dataset.csv")

# Extract the target variable (last column) and flatten to a 1D array
y = df.select(pl.col(df.columns[-1])).to_numpy().flatten()
# Extract features by dropping the last column
x = df.drop(df.columns[-1]).to_numpy()

# assertions
assert x.shape == (1599, 11)
assert y.shape == (1599,)

## $R^2$ Score

Sklearn uses the [$R^2$ score](https://en.wikipedia.org/wiki/Coefficient_of_determination) as a quality measure for regressors. Given true values $y$ and predicted values $\hat{y}$ the $R^2$ score is defined as 

\begin{align*}
R^2(y, \hat{y}) &= 1-\cfrac{\sum_{i=1}^m(y_i-\hat{y}_i)^2}{\sum_{i=1}^m(y_i - \bar{y})^2}\,,
\end{align*}
where $\bar{y}$ is the average of $y$.

This value is 1 if the predictions match exactly, 0 if we would simply always predict the average and negative if our predictions are worse than this simple baseline.\
In short we aim for a value $>0$ and close to $1$.

### Task 2

Implement the $R^2$ score.\
Then use scikit learns [Linear Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) model to fit on the dataset and calculate the $R^2$ score.\
Compare your result to the `.score` method of the regressor.

In [37]:
import numpy.typing as npt
from sklearn.linear_model import LinearRegression


def r2_score(y: npt.NDArray[np.float64], y_hat: npt.NDArray[np.float64]) -> float:
    """Computes the coefficient of determination.

    Parameters
    ----------
    y : npt.NDArray[np.float64]
        True labels, shape (n_samples,).
    y_hat : npt.NDArray[np.float64]
        Predicted labels, shape (n_samples,).

    Returns
    -------
    float
        Score in the range (-inf, 1).
    """

    # Calculate the Residual Sum of Squares (errors made by our predictions)
    rss = np.sum((y - y_hat) ** 2)
    # Calculate the Total Sum of Squares (variance from the mean)
    tss = np.sum((y - np.mean(y)) ** 2)
    
    return 1 - (rss / tss)

regressor = LinearRegression().fit(x, y)
y_hat = regressor.predict(x)
my_score = r2_score(y, y_hat)

print(f"Custom R2 Score:  {my_score:.4f}")
print(f"Sklearn R2 Score: {regressor.score(x, y):.4f}")

# assertions
np.random.seed(0)
p = np.random.rand(100)
p_hat = 2 * p + np.random.rand(100) * 0.001
assert np.isclose(r2_score(p, p_hat), -2.695248533930105)

Custom R2 Score:  0.3606
Sklearn R2 Score: 0.3606


Custom implementation matches sklearn exactly, but this is an in-sample fit (trained and evaluated on the same data), so 0.36 is an optimistic upper bound, not a generalization estimate.

# Stacking

The main idea in stacking is to 
1. learn several heterogenous base models on the original data
2. learn a meta model on the predictions of the base models

<div>
<img src="images/stacking.png" width="600"/>
</div>
The hope is that the meta model can learn to combine the strengths of the base models (e.g. if model 1 fails, model 3 is strong).
Note that in contrast to bagging and boosting the base models must not be of the same method (e.g. decision trees).

## Base Models

First lets select a set of base models. We can now choose from the wide pool of regression methods.

Here we want to use the following models:
- [Linear Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) (with NO extra keywords)
- Polynomial Regression of degree 2 (use a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) of [Polynomial Features](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) followed by Linear Regression) (with NO extra keywords)
- [KNN Regression](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) (with `n_neighbours=10`)
- [Decision Tree Regression](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html) (with `max_depth=4` and `random_state=42`)


### Task 3
Create a list of base models and evaluate them using crossvalidation (avg. over 10 folds).

In [38]:
from sklearn.base import BaseEstimator
from sklearn.model_selection import cross_val_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor

def report_cv(label, scores):
    print(f"{label:<28}: mean R2 = {np.mean(scores):.4f}  std = {np.std(scores):.4f}  "
          f"min = {np.min(scores):.4f}  max = {np.max(scores):.4f}")

# create base models
base_models: list[tuple[str, BaseEstimator]] = [
    ("linear", LinearRegression()),
    ("poly", make_pipeline(PolynomialFeatures(degree=2), LinearRegression())),
    ("knn", KNeighborsRegressor(n_neighbors=10)),
    ("tree", DecisionTreeRegressor(max_depth=4, random_state=42))
]

cross_val_means = {}
base_cv_raw = {}

# Estimate avg. crossvalidation score (10 folds) for each base model
for name, model in base_models:
    # Use R2 as the scoring metric and take the mean across all 10 folds
    cv_scores = cross_val_score(model, x, y, cv=10, scoring='r2')
    cross_val_means[name] = np.mean(cv_scores)
    base_cv_raw[name] = cv_scores
    report_cv(name, cv_scores)

print("Base Models 10-Fold CV R2 Means:", cross_val_means)

linear                      : mean R2 = 0.2355  std = 0.1975  min = -0.2487  max = 0.4032
poly                        : mean R2 = 0.1902  std = 0.1580  min = -0.1878  max = 0.3842
knn                         : mean R2 = 0.0107  std = 0.0824  min = -0.1579  max = 0.1496
tree                        : mean R2 = 0.1634  std = 0.1498  min = -0.1663  max = 0.3476
Base Models 10-Fold CV R2 Means: {'linear': np.float64(0.2355470969430748), 'poly': np.float64(0.1902292193982853), 'knn': np.float64(0.010718379527949207), 'tree': np.float64(0.16338476109004715)}


- `linear: 0.2355 ± 0.1975`: cross-validated performance is much lower and noisier than the in-sample 0.36, the realistic baseline going forward.
- `poly: 0.1902 ± 0.1580`: degree-2 expansion with no regularization does *worse* than plain linear, i.e. it's overfitting.
- `knn: 0.0107 ± 0.0824`: essentially uninformative; consistent with unscaled distances being dominated by large-magnitude features (e.g. sulfur dioxide).
- `tree: 0.1634 ± 0.1498`: mid-pack, with fold variance as large as the other models.

## Meta Model

The meta model uses the predictions of the base models to predict $y$. One can thus view the base models as a feature map for the meta model.

In order to train the meta model, **we need the predictions of the base models on unseen data** since this is the scenario we would face at inference time. A simple method is to use **out-of-fold predictions** during training:

1. separate the data into k folds (deterministically - you don't have to shuffle before).
2. hold out one of the folds and train the base models on the other folds.
3. predict the held out fold using the base models.
4. repeat the above two steps k times to obtain out-of-fold predictions for all k folds.
5. feed all the out-of-fold prediction as features (training data) to the meta model.


### Task 4

Implement the out-of-fold method below.\
Calculate the $R^2$-Score on the out-of-fold predictions for each of the base models.

In [39]:
def oof_prediction(model: BaseEstimator, x: npt.NDArray[np.float64], y: npt.NDArray[np.float64], n_folds: int = 5) -> npt.NDArray[np.float64]:
    """Computes out-of-fold predictions.

    Parameters
    ----------
    model : BaseEstimator
        Model with .fit and .predict methods.
    x : npt.NDArray[np.float64]
        Features, shape (n_samples, n_features).
    y : npt.NDArray[np.float64]
        True labels, shape (n_samples,).
    n_folds : int, optional
        Amount of folds, by default 5.

    Returns
    -------
    npt.NDArray[np.float64]
        Predictions for each sample, such that the model was trained on all folds except the one, which the sample belongs to. Shape (n_samples,).
    """

    # Generate contiguous deterministic folds. np.array_split correctly handles uneven division.
    indices = np.arange(x.shape[0])
    folds_indices = np.array_split(indices, n_folds)
    
    assert np.all([len(fold_indices) >= x.shape[0] // n_folds for fold_indices in folds_indices]), "Some folds are too small."
    assert np.all([len(fold_indices) <= x.shape[0] // n_folds + 1 for fold_indices in folds_indices]), "Some folds are too big."
    assert np.sum([len(fold_indices) for fold_indices in folds_indices]) == x.shape[0], "Either too little or too many indices."
    assert np.all(np.concatenate(folds_indices) == np.arange(x.shape[0])), "Fold indices were permuted."
    
    predictions = np.zeros(x.shape[0])
    
    # Train and predict iteratively holding out one fold at a time
    for i in range(n_folds):
        val_idx = folds_indices[i]
        # Flatten all other fold indices to form the training set
        train_idx = np.concatenate([folds_indices[j] for j in range(n_folds) if j != i])
        
        # Fit on (K-1) folds
        model.fit(x[train_idx], y[train_idx])
        # Predict on the holdout fold
        predictions[val_idx] = model.predict(x[val_idx])
        
    return predictions

from sklearn.base import clone

r2_scores = {}
# Calculate r2 score for oof predictions for each base model
for name, model in base_models:
    # We clone to ensure we do not use an already fitted estimator
    oof_preds = oof_prediction(clone(model), x, y, n_folds=5)
    r2_scores[name] = r2_score(y, oof_preds)

print("Out-of-Fold R2 Scores:", r2_scores)


Out-of-Fold R2 Scores: {'linear': np.float64(0.3300724807706522), 'poly': np.float64(0.2788744306420208), 'knn': np.float64(0.07317948274806763), 'tree': np.float64(0.24850682165565185)}


### Note: Task 3 and Task 4 numbers are not directly comparable

Task 3 reports the **mean of 10 per-fold $R^2$ values** (`cross_val_score` computes 
$R^2$ separately on each held-out fold, using that fold's own local mean as the 
$R^2$ baseline, then we average).

Task 4 reports a **single pooled $R^2$**: all out-of-fold predictions across the 
5 folds are concatenated first, and $R^2$ is


linear 0.330 / poly 0.279 / knn 0.073 / tree 0.249, all higher than their Task 3 counterparts, but as stated above, that's expected from pooled-5-fold-$R^2$ vs mean-of-10-fold-$R^2$ being different statistics, not evidence the models got better.

Now lets put everything together.

### Task 5

Implement the following `Stacking` class. Keep in mind the following things:
- the meta model is trained on out-of-fold predictions of the base models
- the base models are trained on the given dataset
- when predicting, we just use the predictions of the base models (no out-of-fold) as input for the meta model

Use your class to learn a stacked regressor with **linear regression as meta model** and the base models from Task 3. Evaluate it using crossvalidation (avg. of 10 folds) and compare the score to those of the base models (Task 3).

In [40]:
from typing import Self


class StackedRegressor(BaseEstimator):
    def __init__(self, base_models: list[tuple[str, BaseEstimator]], meta_model: BaseEstimator, n_folds: int = 5):
        self.base_models = base_models
        self.meta_model = meta_model
        # n folds used for oof_prediction during training
        self.n_folds = n_folds

    def fit(self, x: npt.NDArray[np.float64], y: npt.NDArray[np.float64]) -> Self:
        """Learns base models and meta model.

        Parameters
        ----------
        x : npt.NDArray[np.float64]
            Features, shape (n_samples, n_features).
        y : npt.NDArray[np.float64]
            True labels, shape (n_samples,).
        """
        
        self.base_models_ = []
        # Matrix to hold out-of-fold features for the meta-model
        oof_features = np.zeros((x.shape[0], len(self.base_models)))

        for i, (name, model) in enumerate(self.base_models):
            # 1. Generate OOF predictions to act as training data for the meta model
            oof_features[:, i] = oof_prediction(clone(model), x, y, n_folds=self.n_folds)
            
            # 2. Fit the actual base model on the entire dataset for inference later
            fitted_model = clone(model).fit(x, y)
            self.base_models_.append((name, fitted_model))

        # 3. Train the meta model on the OOF predictions
        self.meta_model_ = clone(self.meta_model).fit(oof_features, y)
        return self

    def predict(self, x: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
        """Given the features, predict the labels with the base classes, and then predict the label based on the base predictions.

        Parameters
        ----------
        x : npt.NDArray[np.float64]
            Features, shape (n_samples, n_features).

        Returns
        -------
        npt.NDArray[np.float64]
            Predicted label for each sample, shape (n_samples,).
        """

        base_preds = np.zeros((x.shape[0], len(self.base_models_)))
        
        # Gather predictions from each trained base model
        for i, (name, model) in enumerate(self.base_models_):
            base_preds[:, i] = model.predict(x)

        # Meta model outputs the final prediction
        return self.meta_model_.predict(base_preds)

    def score(self, x: npt.NDArray[np.float64], y: npt.NDArray[np.float64]) -> float:
        """Computes the coefficient of determination.

        Parameters
        ----------
        x : npt.NDArray[np.float64]
            Features, shape (n_samples, n_features).
        y : npt.NDArray[np.float64]
            True labels, shape (n_samples,).

        Returns
        -------
        float
            Score in the range (-inf, 1).
        """

        y_hat = self.predict(x)
        return r2_score(y, y_hat)


cloned_bases = [(name, clone(model)) for name, model in base_models]

# Instantiate and fit stacked model
custom_stack = StackedRegressor(
    base_models=cloned_bases, 
    meta_model=LinearRegression(), 
    n_folds=5
)
custom_stack.fit(x, y)

# Evaluate custom stacking model with crossvalidation
custom_cv_scores = cross_val_score(custom_stack, x, y, cv=10, scoring='r2')
cross_val_mean = np.mean(custom_cv_scores)

report_cv("Custom Stacking (10-fold)", custom_cv_scores)

Custom Stacking (10-fold)   : mean R2 = 0.2497  std = 0.1544  min = -0.1233  max = 0.4067


Lands between the linear and poly base scores; its std overlaps heavily with linear's alone, so the gain over the single best base model is small and not clearly significant.

### Task 6

Use the [scikit-learn implementation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html) to learn a stacked regressor.
Evaluate it using crossvalidation (avg. of 10 folds) and compare the score to the scores of task 5.

Note that minor differences can occur due to a more advanced oof-prediction used in sklearn.

In [41]:
from sklearn.ensemble import StackingRegressor

# Fit sklearn stacked model
sklearn_stack = StackingRegressor(
    estimators=[(name, clone(model)) for name, model in base_models],
    final_estimator=LinearRegression(),
    cv=5
)
sklearn_stack.fit(x, y)

# Evaluate with crossvalidation, compare to custom model
sklearn_cv_scores = cross_val_score(sklearn_stack, x, y, cv=10, scoring='r2')
cv_mean_sklearn = np.mean(sklearn_cv_scores)

report_cv("Sklearn Stacking (10-fold)", sklearn_cv_scores)

Sklearn Stacking (10-fold)  : mean R2 = 0.2497  std = 0.1544  min = -0.1233  max = 0.4067


Identical to Task 5's numbers. Validates that the from-scratch `StackedRegressor` is functionally correct.

### Task 7
Try at least two different combinations of regressors for base models and meta model and report the average crossvalidation score.
[Here](https://scikit-learn.org/stable/supervised_learning.html) you can find an overview page of sklearn estimators.

In [42]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Combination 1: Random Forest + K-Neighbors -> Meta: Ridge Regression
combo1_bases = [
    ("rf", RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)),
    ("knn", make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)))
]
combo1_stack = StackingRegressor(estimators=combo1_bases, final_estimator=Ridge())

# Combination 2: Decision Tree + Poly Regression -> Meta: Random Forest
combo2_bases = [
    ("dt", DecisionTreeRegressor(max_depth=5, random_state=42)),
    ("poly", make_pipeline(StandardScaler(), PolynomialFeatures(degree=2), Ridge()))
]
combo2_stack = StackingRegressor(estimators=combo2_bases, final_estimator=RandomForestRegressor(n_estimators=50, random_state=42))

# Evaluate the combinations
c1_raw = cross_val_score(combo1_stack, x, y, cv=10, scoring='r2')
c2_raw = cross_val_score(combo2_stack, x, y, cv=10, scoring='r2')

report_cv("Combination 1 (RF, KNN -> Ridge)", c1_raw)
report_cv("Combination 2 (DT, Poly -> RF)", c2_raw)

Combination 1 (RF, KNN -> Ridge): mean R2 = 0.2790  std = 0.1334  min = -0.0712  max = 0.4237
Combination 2 (DT, Poly -> RF): mean R2 = 0.0343  std = 0.1891  min = -0.4302  max = 0.2778


In [43]:
import polars as pl

summary_rows = []
for name, scores in base_cv_raw.items():
    summary_rows.append({"model": f"base: {name}", "mean_r2": np.mean(scores), "std_r2": np.std(scores)})
summary_rows += [
    {"model": "Custom Stacking",  "mean_r2": np.mean(custom_cv_scores),  "std_r2": np.std(custom_cv_scores)},
    {"model": "Sklearn Stacking", "mean_r2": np.mean(sklearn_cv_scores), "std_r2": np.std(sklearn_cv_scores)},
    {"model": "Combo 1 (RF,KNN->Ridge)", "mean_r2": np.mean(c1_raw), "std_r2": np.std(c1_raw)},
    {"model": "Combo 2 (DT,Poly->RF)",   "mean_r2": np.mean(c2_raw), "std_r2": np.std(c2_raw)},
]

summary = pl.DataFrame(summary_rows).sort("mean_r2", descending=True)
print(summary)

shape: (8, 3)
┌─────────────────────────┬──────────┬──────────┐
│ model                   ┆ mean_r2  ┆ std_r2   │
│ ---                     ┆ ---      ┆ ---      │
│ str                     ┆ f64      ┆ f64      │
╞═════════════════════════╪══════════╪══════════╡
│ Combo 1 (RF,KNN->Ridge) ┆ 0.278953 ┆ 0.133415 │
│ Custom Stacking         ┆ 0.249653 ┆ 0.15438  │
│ Sklearn Stacking        ┆ 0.249653 ┆ 0.15438  │
│ base: linear            ┆ 0.235547 ┆ 0.19753  │
│ base: poly              ┆ 0.190229 ┆ 0.157972 │
│ base: tree              ┆ 0.163385 ┆ 0.149846 │
│ Combo 2 (DT,Poly->RF)   ┆ 0.03425  ┆ 0.189067 │
│ base: knn               ┆ 0.010718 ┆ 0.082404 │
└─────────────────────────┴──────────┴──────────┘


Ranks Combo1 > stacks $\approx$ linear > poly > tree > Combo2 > knn by mean. Most adjacent gaps are smaller than each model's own std, only Combo1's lead at the top and knn/Combo2's weakness at the bottom look like real differences. The middle ordering is within noise.